In [ ]:
# %% [markdown]
# # h5adify simulation benchmark notebook — UPDATED (sex genes + sex×batch effects + batch correction metrics + LLM run)
#
# Adds:
# - 5–10 chrX + 5–10 chrY genes per species (human/mouse) for sex inference robustness
# - sex-dependent batch effects (batch×sex interaction on autosomal genes)
# - extra batch correction metrics (silhouette, kNN mixing entropy, sex-stratified batch silhouette)
# - runs both no-LLM and LLM metadata harmonization (if configured)
# - wider/better plots

# %% =========================
# Cell 0 — paths + dependencies
# =========================
import os
import sys
import zipfile
import subprocess
from pathlib import Path

# --- EDIT THIS ---
ZIP_PATH = Path("h5adify_v0.0.7_final.zip")  # <-- path to your zip

OUTDIR = Path("sim_h5adify_benchmark").resolve()
OUTDIR.mkdir(parents=True, exist_ok=True)

# Optional: run LLM-based metadata harmonization too (requires h5adify LLM backend to be configured)
RUN_LLM = True

# Speed knobs (metrics can be expensive)
METRICS_SUBSAMPLE = 9000     # set None to use all cells
KNN_K = 30                   # for mixing/purity metrics
RANDOM_SEED = 42

# Isolate h5adify state (vocab/prompts/cache) from your main install
os.environ["H5ADIFY_HOME"] = str(OUTDIR / "_h5adify_home")
os.environ["H5ADIFY_CACHE_DIR"] = str(OUTDIR / "_h5adify_cache")

def _ensure_import(pkg: str, pip_name: str | None = None):
    try:
        __import__(pkg)
        return
    except Exception:
        pass
    pip_name = pip_name or pkg
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pip_name])

# Minimal stack
_ensure_import("numpy")
_ensure_import("pandas")
_ensure_import("scipy")
_ensure_import("sklearn", "scikit-learn")
_ensure_import("matplotlib")
_ensure_import("anndata")

# Optional but recommended for UMAP / ComBat in h5adify
try:
    import scanpy  # noqa: F401
except Exception:
    print("[WARN] scanpy not found; installing scanpy (recommended for UMAP/ComBat).")
    _ensure_import("scanpy")

# %% ===================================
# Cell 1 — load h5adify from your ZIP
# ===================================
EXTRACT_DIR = OUTDIR / "_h5adify_extracted"
PKG_ROOT = EXTRACT_DIR / "h5adify_release"

if not PKG_ROOT.exists():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)

if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

import numpy as np
import pandas as pd
from scipy import sparse
import scipy.sparse as sp
import matplotlib.pyplot as plt
import anndata as ad

import h5adify
from h5adify.core import (
    harmonize_anndata,
    harmonize_metadata,
    merge_datasets,
    batch_correct_merged,
)

print("[OK] h5adify version:", getattr(h5adify, "__version__", "unknown"))
print("[OK] Outputs ->", OUTDIR)

# %% =========================================
# Cell 2 — helpers (H5AD-safe + ZINB + genes + metrics)
# =========================================
from sklearn.metrics import adjusted_rand_score, accuracy_score, f1_score, confusion_matrix, silhouette_score
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

RNG = np.random.default_rng(RANDOM_SEED)

def pick_obs_key(adata: ad.AnnData, candidates: list[str]) -> str | None:
    """Return first candidate present in adata.obs, else None."""
    for k in candidates:
        if k in adata.obs.columns:
            return k
    return None

def ensure_obs_alias(
    adata: ad.AnnData,
    alias: str,
    candidates: list[str],
    *,
    cast_str: bool = True,
) -> str | None:
    """
    Ensure adata.obs[alias] exists by copying from the first available candidate.
    Returns alias if created/existing, otherwise None.
    """
    if alias in adata.obs.columns:
        if cast_str:
            adata.obs[alias] = adata.obs[alias].astype(str)
        return alias

    src = pick_obs_key(adata, candidates)
    if src is None:
        return None

    adata.obs[alias] = adata.obs[src].astype(str) if cast_str else adata.obs[src].copy()
    return alias

def filter_zero_variance_genes(a: ad.AnnData, eps: float = 0.0) -> ad.AnnData:
    """
    Optional: remove genes with (near) zero variance to reduce ComBat numerical issues.
    Works for sparse or dense.
    """
    X = a.X
    if sp.issparse(X):
        mean = np.asarray(X.mean(axis=0)).ravel()
        mean2 = np.asarray(X.power(2).mean(axis=0)).ravel()
        var = mean2 - mean**2
    else:
        var = np.var(X, axis=0)

    keep = var > eps
    if np.sum(keep) < 50:
        return a  # don't over-prune
    return a[:, keep].copy()


# --------- H5AD-safe helper ----------
def _none_to_empty(x):
    return "" if x is None else x

def _to_h5ad_safe(obj):
    if obj is None:
        return ""
    if isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, pd.Series):
        return [_to_h5ad_safe(v) for v in obj.astype(object).where(~obj.isna(), None).tolist()]
    if isinstance(obj, pd.DataFrame):
        df = obj.copy().astype(object).where(~obj.isna(), None)
        df = df.astype(str).replace({"None": ""})
        return df.to_csv(sep="\t", index=False)
    if isinstance(obj, dict):
        return {str(k): _to_h5ad_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set, np.ndarray)):
        lst = [_to_h5ad_safe(v) for v in list(obj)]
        base_types = set(type(v) for v in lst)
        if len(base_types) > 1:
            lst = ["" if v is None else str(v) for v in lst]
        lst = [_none_to_empty(v) for v in lst]
        return lst
    return str(obj)

# --------- ZINB sampling ----------
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def zinb_counts(mu: np.ndarray, theta: np.ndarray, alpha0=1.5, alpha1=-1.0) -> np.ndarray:
    scale = mu / np.maximum(theta[None, :], 1e-8)
    lam = RNG.gamma(shape=np.maximum(theta[None, :], 1e-6), scale=np.maximum(scale, 1e-10))
    x = RNG.poisson(lam).astype(np.int32)
    pi = sigmoid(alpha0 + alpha1 * np.log1p(mu))
    drop = RNG.random(size=x.shape) < pi
    x[drop] = 0
    return x

# --------- gene naming heuristics ----------
def titlecase_mouse_symbol(human_symbol: str) -> str:
    s = str(human_symbol)
    if not s:
        return s
    s = s.upper()
    out, first_done = [], False
    for ch in s:
        if ch.isalpha():
            if not first_done:
                out.append(ch.upper()); first_done = True
            else:
                out.append(ch.lower())
        else:
            out.append(ch)
    return "".join(out)

# --------- sex gene sets (5–10 X + 5–10 Y each species) ----------
HUMAN_SEX_GENES_X = [
    "XIST","JPX","RPS4X","DDX3X","EIF2S3","KDM6A","TSIX","ATRX","KDM5C","SMC1A"
]
HUMAN_SEX_GENES_Y = [
    "RPS4Y1","KDM5D","DDX3Y","EIF1AY","UTY","ZFY","PRKY","TMSB4Y","NLGN4Y","USP9Y"
]

MOUSE_SEX_GENES_X = [
    "Xist","Jpx","Rps4x","Ddx3x","Eif2s3x","Kdm6a","Atrx","Kdm5c","Smc1a","Eif2s3x"
]
MOUSE_SEX_GENES_Y = [
    "Ddx3y","Eif2s3y","Kdm5d","Uty","Rps4y1","Zfy1","Zfy2","Usp9y","Sry","Kdm5d"
]

# If you want *exactly* 5–10, keep these lists length ~10 and they’ll be forced into the gene lists.

# --------- marker dictionaries ----------
HUMAN_BRAIN_CELLTYPES = [
    "Excitatory neuron","Inhibitory neuron","Astrocyte","Oligodendrocyte","OPC",
    "Microglia","Endothelial","Pericyte"
]

HUMAN_MARKERS = {
    "Excitatory neuron": ["SLC17A7","SATB2","RBFOX3","CAMK2A","GRIN1"],
    "Inhibitory neuron": ["GAD1","GAD2","SLC6A1","DLX1","DLX2"],
    "Astrocyte": ["GFAP","AQP4","ALDH1L1","SLC1A2","S100B"],
    "Oligodendrocyte": ["MBP","PLP1","MOG","MAG","CLDN11"],
    "OPC": ["PDGFRA","CSPG4","VCAN","SOX10","OLIG1"],
    "Microglia": ["CX3CR1","P2RY12","AIF1","TYROBP","C1QA"],
    "Endothelial": ["PECAM1","VWF","KDR","FLT1","CLDN5"],
    "Pericyte": ["PDGFRB","RGS5","CSPG4","MCAM","DES"],
}

GBM_HUMAN_TUMOR_MARKERS = ["EGFR","PDGFRA","SOX2","OLIG2","MKI67","TOP2A","CDK1","ALDH1A3","CHI3L1","VIM","CD44"]
GBM_NORMAL_HUMAN_MARKERS = ["PTPRC","CD3D","MS4A1","LYZ","NKG7","COL1A1","PECAM1","VWF"]
GBM_MOUSE_HOST_MARKERS = [titlecase_mouse_symbol(x) for x in ["PTPRC","CX3CR1","P2RY12","AIF1","PECAM1","COL1A1","VWF","LYZ"]]

ORTHO_MOUSE2HUMAN_SPOTCHECK = {
    "Trp53":"TP53","Egfr":"EGFR","Pdgfra":"PDGFRA","Mki67":"MKI67","Cx3cr1":"CX3CR1",
    "P2ry12":"P2RY12","Pecam1":"PECAM1","Vwf":"VWF","Col1a1":"COL1A1","Xist":"XIST",
}

# --------- gene list builders ----------
def build_gene_list_human(n_genes: int, force_include: list[str]) -> list[str]:
    base = [
        "ACTB","GAPDH","RPLP0","B2M","MALAT1","EEF1A1","TMSB10","HPRT1","PPIA","RPS18",
        "MT-CO1","MT-CO2","MT-CO3","MT-ND1","MT-ND2","MT-ND3","MT-ND4","MT-ND4L","MT-ND5","MT-ND6",
        "MT-ATP6","MT-ATP8","MT-CYB",
        "HLA-A","HLA-B","HLA-C","HLA-DRA","HLA-DRB1",
    ]
    base += [f"RPL{i}" for i in range(3, 41)]
    base += [f"RPS{i}" for i in range(3, 31)]
    base += [f"ZNF{i}" for i in range(1, 401)]

    genes = list(dict.fromkeys(base + force_include))
    while len(genes) < n_genes:
        choice = RNG.integers(0, 4)
        if choice == 0:
            g = f"SLC{RNG.integers(1, 60)}A{RNG.integers(1, 20)}"
        elif choice == 1:
            g = f"KRT{RNG.integers(1, 25)}"
        elif choice == 2:
            g = f"DHX{RNG.integers(1, 60)}"
        else:
            g = f"FNDC{RNG.integers(1, 20)}"
        if g not in genes:
            genes.append(g)
    return genes[:n_genes]

def build_gene_list_mouse_from_human(human_genes: list[str], add_mouse_only: list[str] | None = None) -> list[str]:
    mg = [titlecase_mouse_symbol(g) if not g.startswith("MT-") else g.lower() for g in human_genes]
    add_mouse_only = add_mouse_only or []
    for g in add_mouse_only:
        if g not in mg:
            mg.append(g)
    return mg

# --------- simulation math helpers ----------
def make_size_factors(n: int, mean_log=0.0, sigma=0.5):
    return RNG.lognormal(mean=mean_log, sigma=sigma, size=n).astype(np.float32)

def add_multiplicative_effects(mat, gene_idx, strength=0.6):
    mult = np.ones(mat.shape[1], dtype=np.float32)
    mult[gene_idx] = np.exp(RNG.normal(0, strength, size=len(gene_idx))).astype(np.float32)
    return mat * mult[None, :], mult

def make_fc_matrix(celltypes: np.ndarray, genes: list[str], marker_dict: dict[str, list[str]], fc=6.0):
    g2i = {g:i for i,g in enumerate(genes)}
    F = np.ones((celltypes.shape[0], len(genes)), dtype=np.float32)
    for ct, markers in marker_dict.items():
        rows = np.where(celltypes == ct)[0]
        if rows.size == 0:
            continue
        cols = [g2i[m] for m in markers if m in g2i]
        if cols:
            F[np.ix_(rows, cols)] *= fc
    return F

def make_sex_fc_human(sex: np.ndarray, genes: list[str], female_fc=9.0, male_fc=11.0):
    g2i = {g:i for i,g in enumerate(genes)}
    F = np.ones((sex.shape[0], len(genes)), dtype=np.float32)

    # female X up
    for xg in HUMAN_SEX_GENES_X:
        if xg in g2i:
            F[np.ix_(np.where(sex=="female")[0], [g2i[xg]])] *= female_fc

    # male Y up
    for yg in HUMAN_SEX_GENES_Y:
        if yg in g2i:
            F[np.ix_(np.where(sex=="male")[0], [g2i[yg]])] *= male_fc

    return F

def make_sex_fc_mouse(sex: np.ndarray, genes: list[str], female_fc=9.0, male_fc=11.0):
    g2i = {g:i for i,g in enumerate(genes)}
    F = np.ones((sex.shape[0], len(genes)), dtype=np.float32)

    for xg in MOUSE_SEX_GENES_X:
        if xg in g2i:
            F[np.ix_(np.where(sex=="female")[0], [g2i[xg]])] *= female_fc

    for yg in MOUSE_SEX_GENES_Y:
        if yg in g2i:
            F[np.ix_(np.where(sex=="male")[0], [g2i[yg]])] *= male_fc

    return F

def make_sex_dependent_batch_effect(
    sex: np.ndarray,
    batch: np.ndarray,
    n_genes: int,
    n_interaction_genes: int = 80,
    strength: float = 0.6,
):
    """
    Create a batch×sex interaction multiplier on a random subset of genes.
    This simulates a confounded technical artefact correlated with sex.
    """
    g_idx = RNG.choice(n_genes, size=min(n_interaction_genes, n_genes), replace=False)
    mult = np.ones((len(sex), n_genes), dtype=np.float32)

    for b in np.unique(batch):
        for s in ["female","male"]:
            rows = np.where((batch == b) & (sex == s))[0]
            if rows.size == 0:
                continue
            tmp = np.ones((rows.size, n_genes), dtype=np.float32)
            # different random direction per (batch, sex)
            tmp, _ = add_multiplicative_effects(tmp, g_idx, strength=strength)
            # small systematic tilt to make sex-associated batch effects “real”
            tilt = 1.0 + (0.08 if s == "female" else -0.08)
            tmp[:, g_idx] *= tilt
            mult[rows, :] = tmp

    return mult

def simulate_counts_zinb(
    n_obs: int,
    genes: list[str],
    base_mean_scale: float,
    size_factors: np.ndarray,
    fc_mats: list[np.ndarray] | None = None,
    mult_mats: list[np.ndarray] | None = None,
    theta_shape=2.0,
    theta_scale=2.0,
):
    n_genes = len(genes)
    base_mean = RNG.gamma(shape=0.4, scale=base_mean_scale, size=n_genes).astype(np.float32)
    theta = RNG.gamma(shape=theta_shape, scale=theta_scale, size=n_genes).astype(np.float32)

    mu = (size_factors[:, None] * base_mean[None, :]).astype(np.float32)

    if mult_mats:
        for M in mult_mats:
            mu *= M

    if fc_mats:
        for F in fc_mats:
            mu *= F

    x = zinb_counts(mu=mu, theta=theta, alpha0=1.5, alpha1=-1.0)
    return x

def collapse_duplicate_varnames_sum(adata: ad.AnnData) -> ad.AnnData:
    vn = pd.Index([str(v) for v in adata.var_names])
    if not vn.has_duplicates:
        return adata

    codes, uniques = pd.factorize(vn, sort=False)
    n_vars = len(vn)
    n_u = len(uniques)

    M = sp.csr_matrix(
        (np.ones(n_vars, dtype=np.float32), (np.arange(n_vars), codes)),
        shape=(n_vars, n_u),
    )

    def _matmul_X(X):
        if sp.issparse(X):
            return X @ M
        return X.dot(M.toarray())

    X_new = _matmul_X(adata.X)

    new_var = pd.DataFrame(index=pd.Index(uniques, name=adata.var_names.name or "gene"))
    if adata.var is not None and adata.var.shape[1] > 0:
        tmp = adata.var.copy()
        tmp.index = vn
        for col in tmp.columns:
            new_var[col] = tmp[col].groupby(level=0).first().reindex(uniques).values

    adata2 = ad.AnnData(
        X=X_new,
        obs=adata.obs.copy(),
        var=new_var,
        uns=adata.uns.copy(),
        obsm=adata.obsm.copy(),
        varm=adata.varm.copy(),
        obsp=adata.obsp.copy(),
        varp=adata.varp.copy(),
    )
    for lk, L in adata.layers.items():
        adata2.layers[lk] = _matmul_X(L)
    return adata2

# --------- evaluation helpers ----------
def _get_pred_col(adata: ad.AnnData, field: str) -> str | None:
    c1 = f"h5adify_{field}"
    if c1 in adata.obs.columns:
        return c1
    if field in adata.obs.columns:
        return field
    return None

def evaluate_metadata_fields(raw: ad.AnnData, harm: ad.AnnData, fields: list[str]):
    rows = []
    for f in fields:
        truth = raw.obs.get(f"truth_{f}", None)
        pred_col = _get_pred_col(harm, f)
        if truth is None or pred_col is None:
            rows.append({"field": f, "ari": np.nan, "acc": np.nan, "pred_col": pred_col})
            continue
        pred = harm.obs[pred_col].astype(str).values
        tru = truth.astype(str).values
        rows.append({
            "field": f,
            "ari": adjusted_rand_score(tru, pred),
            "acc": accuracy_score(tru, pred),
            "pred_col": pred_col
        })
    return pd.DataFrame(rows)

def evaluate_sex(raw: ad.AnnData, harm: ad.AnnData):
    truth = raw.obs.get("truth_sex", None)
    pred_col = _get_pred_col(harm, "sex")
    if truth is None or pred_col is None:
        return None, None
    y_true = truth.astype(str).values
    y_pred = harm.obs[pred_col].astype(str).values
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=["female","male","unknown"])
    return acc, cm

def gene_spotcheck_orthologs(harm: ad.AnnData, mapping: dict[str,str]):
    vn = set(map(str, harm.var_names))
    hits = []
    for m, h in mapping.items():
        hits.append({"mouse_symbol": m, "expected_human": h, "present_expected": (h in vn), "present_mouse": (m in vn)})
    return pd.DataFrame(hits)

def summarize_gene_namespace(harm: ad.AnnData):
    vn = pd.Index(map(str, harm.var_names))
    frac_upper = np.mean([g.isupper() and any(ch.isalpha() for ch in g) for g in vn])
    frac_title = np.mean([(len(g)>1 and g[0].isupper() and any(ch.islower() for ch in g[1:])) for g in vn])
    dup = vn.duplicated().mean()
    return {"n_genes": len(vn), "frac_allcaps": float(frac_upper), "frac_titlecase": float(frac_title), "dup_frac": float(dup)}

def _subsample_idx(n, max_n=None):
    if max_n is None or n <= max_n:
        return np.arange(n)
    return RNG.choice(n, size=max_n, replace=False)

def embed_svd(adata: ad.AnnData, n_components=50):
    X = adata.X
    svd = TruncatedSVD(n_components=min(n_components, adata.n_vars - 1), random_state=0)
    Z = svd.fit_transform(X)
    return Z

def knn_batch_mixing_entropy(Z, batch_labels, k=30):
    """
    Higher is better (more mixing). Normalized to [0,1] by log(#batches).
    """
    batch_labels = np.asarray(batch_labels).astype(str)
    uniq = np.unique(batch_labels)
    nb = len(uniq)
    if nb <= 1:
        return np.nan

    nn = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
    nn.fit(Z)
    idx = nn.kneighbors(Z, return_distance=False)[:, 1:]  # drop self

    ent = []
    denom = np.log(nb)
    for i in range(Z.shape[0]):
        neigh = batch_labels[idx[i]]
        counts = np.array([(neigh == b).mean() for b in uniq], dtype=float)
        counts = counts[counts > 0]
        e = -np.sum(counts * np.log(counts))
        ent.append(e / denom if denom > 0 else np.nan)
    return float(np.mean(ent))

def knn_label_purity(Z, labels, k=30):
    """
    Higher means neighbors share same label (bio separation).
    """
    labels = np.asarray(labels).astype(str)
    nn = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
    nn.fit(Z)
    idx = nn.kneighbors(Z, return_distance=False)[:, 1:]
    pur = []
    for i in range(Z.shape[0]):
        neigh = labels[idx[i]]
        pur.append(float(np.mean(neigh == labels[i])))
    return float(np.mean(pur))

def safe_silhouette(Z, labels):
    labels = np.asarray(labels).astype(str)
    if len(np.unique(labels)) < 2:
        return np.nan
    # silhouette_score can crash if a class has 1 sample; guard lightly:
    vc = pd.Series(labels).value_counts()
    if (vc < 2).any():
        return np.nan
    return float(silhouette_score(Z, labels, metric="euclidean"))

def batch_correction_metrics(adata: ad.AnnData, batch_key: str, bio_key: str | None, sex_key: str | None, k=30, subsample=9000):
    """
    Compute metrics in a fixed embedding (SVD on log-normalized X).
    Returns dict.
    """
    obs = adata.obs
    if batch_key not in obs.columns:
        return {}

    idx = _subsample_idx(adata.n_obs, subsample)
    a = adata[idx].copy()

    Z = embed_svd(a, n_components=50)

    out = {}
    out["sil_batch"] = safe_silhouette(Z, a.obs[batch_key].astype(str).values)

    if bio_key is not None and bio_key in a.obs.columns:
        out["sil_bio"] = safe_silhouette(Z, a.obs[bio_key].astype(str).values)
        out["knn_bio_purity"] = knn_label_purity(Z, a.obs[bio_key].astype(str).values, k=k)
    else:
        out["sil_bio"] = np.nan
        out["knn_bio_purity"] = np.nan

    out["knn_batch_mixing"] = knn_batch_mixing_entropy(Z, a.obs[batch_key].astype(str).values, k=k)

    # sex-stratified batch silhouette (captures sex-dependent batch effects)
    if sex_key is not None and sex_key in a.obs.columns:
        for s in ["female","male"]:
            rows = np.where(a.obs[sex_key].astype(str).values == s)[0]
            if rows.size < 200:
                out[f"sil_batch_{s}"] = np.nan
            else:
                out[f"sil_batch_{s}"] = safe_silhouette(Z[rows], a.obs[batch_key].astype(str).values[rows])
    else:
        out["sil_batch_female"] = np.nan
        out["sil_batch_male"] = np.nan

    return out

# %% =========================================================
# Cell 3 — Simulation A: Human physiological brain scRNA (3 studies) with sex genes + sex×batch effects
# =========================================================
def simulate_human_brain_scrna_replicates(outdir: Path, n_datasets=3, n_cells_each=4000, n_genes=900):
    out_paths = []
    celltypes = np.array(HUMAN_BRAIN_CELLTYPES, dtype=object)
    ct_probs = np.array([0.22,0.18,0.12,0.14,0.10,0.10,0.08,0.06], dtype=np.float32)
    ct_probs /= ct_probs.sum()

    force_genes = sorted({g for v in HUMAN_MARKERS.values() for g in v} | set(HUMAN_SEX_GENES_X) | set(HUMAN_SEX_GENES_Y))
    genes = build_gene_list_human(n_genes=n_genes, force_include=force_genes)

    for k in range(n_datasets):
        ct = RNG.choice(celltypes, size=n_cells_each, p=ct_probs)
        donors = RNG.choice([f"DONOR_{k}_A", f"DONOR_{k}_B", f"DONOR_{k}_C"], size=n_cells_each, p=[0.4,0.35,0.25])
        batches = RNG.choice([f"batch{k}_1", f"batch{k}_2"], size=n_cells_each, p=[0.55,0.45])
        domains = RNG.choice(["CTX_L2/3","CTX_L4","HIP_CA1","STR"], size=n_cells_each, p=[0.35,0.25,0.25,0.15])
        samples = np.array([f"S{k}_{d}_{b}" for d,b in zip(donors,batches)], dtype=object)

        donor2sex = {d: RNG.choice(["female","male"]) for d in np.unique(donors)}
        sex = np.array([donor2sex[d] for d in donors], dtype=object)

        # size factors with batch depth
        sf = make_size_factors(n_cells_each, sigma=0.6)
        sf *= np.where(batches == f"batch{k}_1", 1.0, 1.25).astype(np.float32)

        # biological FC
        F_ct  = make_fc_matrix(ct, genes, HUMAN_MARKERS, fc=7.0)
        F_sex = make_sex_fc_human(sex, genes, female_fc=9.0, male_fc=11.0)

        # batch + domain effects
        g_idx_batch  = RNG.choice(len(genes), size=60, replace=False)
        g_idx_domain = RNG.choice(len(genes), size=50, replace=False)

        batch_mult = np.ones((n_cells_each, len(genes)), dtype=np.float32)
        domain_mult = np.ones((n_cells_each, len(genes)), dtype=np.float32)

        for b in np.unique(batches):
            rows = np.where(batches == b)[0]
            tmp = np.ones((len(rows), len(genes)), dtype=np.float32)
            tmp, _ = add_multiplicative_effects(tmp, g_idx_batch, strength=0.55)
            batch_mult[rows, :] = tmp

        for dmn in np.unique(domains):
            rows = np.where(domains == dmn)[0]
            tmp = np.ones((len(rows), len(genes)), dtype=np.float32)
            tmp, _ = add_multiplicative_effects(tmp, g_idx_domain, strength=0.45)
            domain_mult[rows, :] = tmp

        # NEW: sex-dependent batch effect (batch×sex interaction)
        sex_batch_mult = make_sex_dependent_batch_effect(
            sex=sex, batch=batches, n_genes=len(genes), n_interaction_genes=85, strength=0.55
        )

        X = simulate_counts_zinb(
            n_obs=n_cells_each,
            genes=genes,
            base_mean_scale=2.0,
            size_factors=sf,
            fc_mats=[F_ct, F_sex],
            mult_mats=[batch_mult, domain_mult, sex_batch_mult],
        )

        adata = ad.AnnData(
            X=sparse.csr_matrix(X),
            obs=pd.DataFrame(index=[f"cell_{k}_{i}" for i in range(n_cells_each)]),
            var=pd.DataFrame(index=pd.Index(genes, name="gene")),
        )

        # truth
        adata.obs["truth_celltype"] = ct
        adata.obs["truth_donor"] = donors
        adata.obs["truth_batch"] = batches
        adata.obs["truth_domain"] = domains
        adata.obs["truth_sample"] = samples
        adata.obs["truth_species"] = "human"
        adata.obs["truth_technology"] = "10x-3prime"
        adata.obs["truth_sex"] = sex

        # messy metadata
        if k == 0:
            adata.obs["patient"] = donors
            adata.obs["library"] = batches
            adata.obs["brain_region"] = domains
            adata.obs["sample_name"] = samples
            adata.obs["gender"] = np.where(RNG.random(n_cells_each) < 0.7, sex, "unknown")
            adata.obs["organism"] = "Homo sapiens"
            adata.obs["platform"] = "10x 3' v3"
        elif k == 1:
            adata.obs["donor_id"] = donors
            adata.obs["batch_id"] = batches
            adata.obs["Region"] = domains
            adata.obs["sample_id"] = samples
            adata.obs["sex"] = np.where(RNG.random(n_cells_each) < 0.5, sex, "NA")
            adata.obs["species"] = "human"
            adata.obs["assay"] = "Chromium Single Cell 3' v3"
        else:
            adata.obs["subject"] = donors
            adata.obs["run"] = batches
            adata.obs["tissue_region"] = domains
            adata.obs["biosample"] = samples
            adata.obs["Sex"] = np.where(sex=="female", "0", "1")
            adata.uns["Organism"] = "human"
            adata.obs["tech"] = "10x"

        adata.var["gene_symbols"] = adata.var_names
        adata.var["feature_name"] = adata.var_names

        p = outdir / f"simA_human_brain_study{k+1}.h5ad"
        adata.write_h5ad(p)
        out_paths.append(p)

    return out_paths

simA_paths = simulate_human_brain_scrna_replicates(OUTDIR, n_datasets=3, n_cells_each=4000, n_genes=900)
print("[OK] Simulation A files:", [p.name for p in simA_paths])

# %% =====================================================
# Cell 4 — Simulation B: GBM PDX (human tumor + mouse host) with expanded sex genes + sex×batch effects
# =====================================================
def simulate_gbm_pdx_replicates(outdir: Path, n_datasets=2, n_cells_each=5000, n_genes=1100):
    out_paths = []

    force_h = sorted(set(GBM_HUMAN_TUMOR_MARKERS + GBM_NORMAL_HUMAN_MARKERS + HUMAN_SEX_GENES_X + HUMAN_SEX_GENES_Y))
    human_genes = build_gene_list_human(n_genes=int(n_genes*0.65), force_include=force_h)

    # ensure mouse sex genes are present
    mouse_force = [titlecase_mouse_symbol(g) for g in GBM_MOUSE_HOST_MARKERS] + MOUSE_SEX_GENES_X + MOUSE_SEX_GENES_Y
    mouse_genes = build_gene_list_mouse_from_human(human_genes[:int(n_genes*0.35)], add_mouse_only=mouse_force)

    genes_mixed = list(dict.fromkeys(human_genes + mouse_genes))[:n_genes]

    human_tumor_cts = ["GBM_Malignant_OPC-like", "GBM_Malignant_MES-like", "GBM_Cycling"]
    human_normal_cts = ["Tcell", "Bcell", "Myeloid", "Endothelial", "Fibroblast"]
    mouse_host_cts  = ["Mouse_Microglia", "Mouse_Myeloid", "Mouse_Endothelial", "Mouse_Fibroblast"]

    for k in range(n_datasets):
        species = RNG.choice(["human","mouse"], size=n_cells_each, p=[0.7,0.3])

        neoplastic = np.zeros(n_cells_each, dtype=bool)
        neoplastic[np.where(species=="human")[0]] = (RNG.random(np.sum(species=="human")) < 0.75)

        ct = np.empty(n_cells_each, dtype=object)
        idx_h_t = np.where((species=="human") & (neoplastic))[0]
        ct[idx_h_t] = RNG.choice(human_tumor_cts, size=idx_h_t.size, p=[0.45,0.35,0.20])
        idx_h_n = np.where((species=="human") & (~neoplastic))[0]
        ct[idx_h_n] = RNG.choice(human_normal_cts, size=idx_h_n.size, p=[0.25,0.10,0.35,0.20,0.10])
        idx_m = np.where(species=="mouse")[0]
        ct[idx_m] = RNG.choice(mouse_host_cts, size=idx_m.size, p=[0.55,0.15,0.20,0.10])

        donor = RNG.choice([f"PDX{100+k}_M{j}" for j in range(1,4)], size=n_cells_each, p=[0.5,0.3,0.2])
        batch = RNG.choice([f"pdx{k}_batch1", f"pdx{k}_batch2"], size=n_cells_each, p=[0.6,0.4])
        sample = np.array([f"PDX{k}_TP{RNG.choice([0,7,14])}_{d}" for d in donor], dtype=object)
        tech = RNG.choice(["10x-5prime","10x-3prime"], size=n_cells_each, p=[0.4,0.6])

        donor2sex = {d: RNG.choice(["female","male"]) for d in np.unique(donor)}
        sex = np.array([donor2sex[d] for d in donor], dtype=object)

        sf = make_size_factors(n_cells_each, sigma=0.7)
        sf *= np.where(neoplastic, 1.35, 1.0).astype(np.float32)

        marker_map = {
            "GBM_Malignant_OPC-like": ["OLIG2","SOX2","PDGFRA"],
            "GBM_Malignant_MES-like": ["VIM","CD44","CHI3L1","ALDH1A3"],
            "GBM_Cycling": ["MKI67","TOP2A","CDK1"],
            "Tcell": ["CD3D","NKG7"],
            "Bcell": ["MS4A1"],
            "Myeloid": ["LYZ","PTPRC"],
            "Endothelial": ["PECAM1","VWF"],
            "Fibroblast": ["COL1A1"],
            "Mouse_Microglia": ["Cx3cr1","P2ry12","Aif1"],
            "Mouse_Myeloid": ["Lyz","Ptprc"],
            "Mouse_Endothelial": ["Pecam1","Vwf"],
            "Mouse_Fibroblast": ["Col1a1"],
        }
        F_ct = make_fc_matrix(ct, genes_mixed, marker_map, fc=8.0)

        # sex FC by species
        F_sex = np.ones((n_cells_each, len(genes_mixed)), dtype=np.float32)
        g2i = {g:i for i,g in enumerate(genes_mixed)}
        # human genes
        for xg in HUMAN_SEX_GENES_X:
            if xg in g2i:
                F_sex[np.ix_(np.where((species=="human") & (sex=="female"))[0], [g2i[xg]])] *= 9.0
        for yg in HUMAN_SEX_GENES_Y:
            if yg in g2i:
                F_sex[np.ix_(np.where((species=="human") & (sex=="male"))[0], [g2i[yg]])] *= 11.0
        # mouse genes
        for xg in MOUSE_SEX_GENES_X:
            if xg in g2i:
                F_sex[np.ix_(np.where((species=="mouse") & (sex=="female"))[0], [g2i[xg]])] *= 9.0
        for yg in MOUSE_SEX_GENES_Y:
            if yg in g2i:
                F_sex[np.ix_(np.where((species=="mouse") & (sex=="male"))[0], [g2i[yg]])] *= 11.0

        # batch effects + sex×batch effects
        g_idx_batch = RNG.choice(len(genes_mixed), size=90, replace=False)
        batch_mult = np.ones((n_cells_each, len(genes_mixed)), dtype=np.float32)
        for b in np.unique(batch):
            rows = np.where(batch == b)[0]
            tmp = np.ones((len(rows), len(genes_mixed)), dtype=np.float32)
            tmp, _ = add_multiplicative_effects(tmp, g_idx_batch, strength=0.65)
            batch_mult[rows, :] = tmp

        sex_batch_mult = make_sex_dependent_batch_effect(
            sex=sex, batch=batch, n_genes=len(genes_mixed), n_interaction_genes=95, strength=0.65
        )

        X = simulate_counts_zinb(
            n_obs=n_cells_each,
            genes=genes_mixed,
            base_mean_scale=2.2,
            size_factors=sf,
            fc_mats=[F_ct, F_sex],
            mult_mats=[batch_mult, sex_batch_mult],
        )

        adata = ad.AnnData(
            X=sparse.csr_matrix(X),
            obs=pd.DataFrame(index=[f"pdx{k}_cell_{i}" for i in range(n_cells_each)]),
            var=pd.DataFrame(index=pd.Index(genes_mixed, name="gene")),
        )

        adata.obs["truth_celltype"] = ct
        adata.obs["truth_species"] = species
        adata.obs["truth_neoplastic"] = neoplastic.astype(int)
        adata.obs["truth_donor"] = donor
        adata.obs["truth_batch"] = batch
        adata.obs["truth_sample"] = sample
        adata.obs["truth_technology"] = tech
        adata.obs["truth_sex"] = sex

        if k == 0:
            adata.obs["patient_id"] = donor
            adata.obs["processing_batch"] = batch
            adata.obs["biosample"] = sample
            adata.obs["assay"] = tech
            adata.obs["organism"] = np.where(RNG.random(n_cells_each) < 0.85, species, "unknown")
            adata.obs["gender"] = "unknown"
        else:
            adata.obs["donor"] = donor
            adata.obs["lane"] = batch
            adata.obs["sample_id"] = sample
            adata.obs["platform"] = tech
            adata.obs["Species"] = np.where(species=="human", "Homo sapiens", "Mus musculus")
            adata.obs["Sex"] = np.where(sex=="female", "0", "1")

        adata.var["gene_symbols"] = adata.var_names
        adata.var["feature_name"] = adata.var_names
        adata.var["ensembl_like"] = [
            ("ENSG" + str(10_000_000 + i)) if g.isupper() and g.isalnum() else
            ("ENSMUSG" + str(10_000_000 + i)) if (len(g)>1 and g[0].isupper() and any(ch.islower() for ch in g[1:])) else
            ""
            for i,g in enumerate(adata.var_names)
        ]
        adata.uns["truth_mouse2human_spotcheck"] = ORTHO_MOUSE2HUMAN_SPOTCHECK

        p = outdir / f"simB_gbm_pdx_{k+1}.h5ad"
        adata.write_h5ad(p)
        out_paths.append(p)

    return out_paths

simB_paths = simulate_gbm_pdx_replicates(OUTDIR, n_datasets=2, n_cells_each=5000, n_genes=1100)
print("[OK] Simulation B files:", [p.name for p in simB_paths])

# %% ============================================================
# Cell 5 — Simulation C: Mouse brain spatial (multi-tech) with expanded sex genes + sex×batch effects
# ============================================================
def simulate_mouse_spatial_multitech(outdir: Path, n_sections=3, spots_per_tech=1200, n_genes=900):
    out_paths = []

    # build human-looking genes then mouse-case them; force mouse sex genes
    force_h = sorted({g for v in HUMAN_MARKERS.values() for g in v} | set(HUMAN_SEX_GENES_X) | set(HUMAN_SEX_GENES_Y))
    human_genes = build_gene_list_human(n_genes=n_genes, force_include=force_h)
    mouse_genes = build_gene_list_mouse_from_human(human_genes, add_mouse_only=(MOUSE_SEX_GENES_X + MOUSE_SEX_GENES_Y))

    mouse_region_markers = {
        "cortex": ["Reln","Slc17a7","Gad1","Gad2","Aldh1l1"],
        "hippocampus": ["Prox1","Calb1","Grin1","Slc1a2","Aqp4"],
        "striatum": ["Ppp1r1b","Drd1","Drd2","Gad1","Gad2"],
    }
    for ms in mouse_region_markers.values():
        for g in ms:
            if g not in mouse_genes:
                mouse_genes.append(g)
    mouse_genes = mouse_genes[:n_genes]

    techs = [("10x-Visium", 1.5), ("Stereo-seq", 1.2), ("Slide-seqV2", 0.8)]

    for sec in range(1, n_sections+1):
        sex = RNG.choice(["female","male"])
        donor = f"MOUSE_{sec:02d}"
        batch = RNG.choice([f"sp_batch{sec}_1", f"sp_batch{sec}_2"])

        for tech, depth_mult in techs:
            n_spots = spots_per_tech
            x = RNG.uniform(0, 1, size=n_spots)
            y = RNG.uniform(0, 1, size=n_spots)
            coords = np.stack([x, y], axis=1).astype(np.float32)

            region = np.where(x < 0.33, "cortex", np.where(x < 0.66, "hippocampus", "striatum")).astype(object)
            domain = np.where(y < 0.5, region.astype(str) + "_deep", region.astype(str) + "_sup").astype(object)

            sf = make_size_factors(n_spots, sigma=0.55)
            sf *= (2500.0 * depth_mult)
            sf /= np.median(sf)

            # region FC
            F_region = np.ones((n_spots, len(mouse_genes)), dtype=np.float32)
            g2i = {g:i for i,g in enumerate(mouse_genes)}
            for r, ms in mouse_region_markers.items():
                rows = np.where(region == r)[0]
                cols = [g2i[m] for m in ms if m in g2i]
                if cols:
                    F_region[np.ix_(rows, cols)] *= 6.5

            # sex FC (expanded sets)
            sex_arr = np.array([sex]*n_spots, dtype=object)
            F_sex = make_sex_fc_mouse(sex_arr, mouse_genes, female_fc=9.0, male_fc=11.0)

            # tech batch effect
            g_idx_tech = RNG.choice(len(mouse_genes), size=75, replace=False)
            tech_mult = np.ones((n_spots, len(mouse_genes)), dtype=np.float32)
            tech_mult, _ = add_multiplicative_effects(tech_mult, g_idx_tech, strength=0.55)

            # sex-dependent tech effect (treat "tech" as batch-like; within section sex is constant, still creates sex-associated artifacts across sections)
            tech_as_batch = np.array([tech]*n_spots, dtype=object)
            sex_batch_mult = make_sex_dependent_batch_effect(
                sex=sex_arr, batch=tech_as_batch, n_genes=len(mouse_genes), n_interaction_genes=70, strength=0.55
            )

            X = simulate_counts_zinb(
                n_obs=n_spots,
                genes=mouse_genes,
                base_mean_scale=1.7,
                size_factors=sf,
                fc_mats=[F_region, F_sex],
                mult_mats=[tech_mult, sex_batch_mult],
            )

            adata = ad.AnnData(
                X=sparse.csr_matrix(X),
                obs=pd.DataFrame(index=[f"spot_sec{sec}_{tech}_{i}" for i in range(n_spots)]),
                var=pd.DataFrame(index=pd.Index(mouse_genes, name="gene")),
            )
            adata.obsm["spatial"] = coords

            adata.obs["truth_region"] = region
            adata.obs["truth_domain"] = domain
            adata.obs["truth_batch"] = batch
            adata.obs["truth_sample"] = f"SECTION_{sec}"
            adata.obs["truth_donor"] = donor
            adata.obs["truth_species"] = "mouse"
            adata.obs["truth_technology"] = tech
            adata.obs["truth_sex"] = sex

            # messy metadata
            adata.obs["sample_name"] = f"sec{sec}"
            adata.obs["run"] = batch
            adata.obs["tissue_region"] = region
            adata.obs["ROI"] = domain
            adata.obs["platform"] = tech
            adata.obs["Sex"] = "unknown"
            adata.obs["organism"] = "Mus musculus"

            adata.var["gene_symbols"] = adata.var_names
            adata.var["feature_name"] = adata.var_names

            p = outdir / f"simC_mouse_spatial_sec{sec}_{tech.replace(' ','_').replace('/','_')}.h5ad"
            adata.write_h5ad(p)
            out_paths.append(p)

    return out_paths

simC_paths = simulate_mouse_spatial_multitech(OUTDIR, n_sections=3, spots_per_tech=1200, n_genes=900)
print("[OK] Simulation C files:", len(simC_paths))

# %% ==========================================================
# Cell 6 — Run h5adify harmonization (genes + metadata): no_llm and (optionally) llm
# ==========================================================
from requests.exceptions import RequestException

def run_h5adify_harmonization(
    adata: ad.AnnData,
    target_species: str = "human",
    do_gene_harmonize: bool = True,
    do_meta_harmonize: bool = True,
    use_llm: bool = False,
):
    reports = {}
    adata_h = adata.copy()

    # ---- gene harmonization ----
    if do_gene_harmonize:
        try:
            adata_h, gene_report, profile = harmonize_anndata(
                adata_h,
                target="hugo",
                target_species=target_species,
                var_key=None,
                out_key="gene_harmonized",
                inplace=True,
                deduplicate=True,
                dedup_how="sum",
                chunk_size=800,
            )
            try:
                reports["gene_profile"] = getattr(profile, "to_dict", lambda: profile)()
            except Exception:
                reports["gene_profile"] = str(profile)

            try:
                reports["gene_report_head_tsv"] = gene_report.head(25).to_csv(sep="\t", index=False)
            except Exception:
                reports["gene_report_head_tsv"] = str(gene_report.head(25))

        except RequestException as e:
            reports["gene_harmonize_error"] = f"{type(e).__name__}: {e}"
            vn = pd.Index([str(x) for x in adata_h.var_names])
            adata_h.var["gene_harmonized"] = [str(x).upper() for x in vn]
            adata_h.var_names = pd.Index(adata_h.var["gene_harmonized"])
            reports["gene_profile"] = {"species": "unknown_offline_fallback"}
        except Exception as e:
            reports["gene_harmonize_error"] = f"{type(e).__name__}: {e}"

    # collapse duplicates
    adata_h = collapse_duplicate_varnames_sum(adata_h)

    # ---- metadata harmonization ----
    if do_meta_harmonize:
        try:
            adata_h, meta_report = harmonize_metadata(
                adata_h,
                use_llm=use_llm,
                llm_prompt_name="metadata_harmonize_v1_default",
                sex_from_expression=True,
                min_sex_frac=0.02,
                sex_groupby="auto",
                sex_min_group_obs=25,
                inplace=False,
            )
            # store as TSV string if dataframe-like
            if isinstance(meta_report, pd.DataFrame):
                reports["meta_report_head_tsv"] = meta_report.head(50).to_csv(sep="\t", index=False)
            else:
                reports["meta_report_head_tsv"] = str(meta_report)
        except Exception as e:
            reports["meta_harmonize_error"] = f"{type(e).__name__}: {e}"

    adata_h.uns["h5adify_benchmark_reports"] = _to_h5ad_safe(reports)
    return adata_h

all_paths = simA_paths + simB_paths + simC_paths

def harmonize_all(method_name: str, use_llm: bool):
    outdir = OUTDIR / f"harm_{method_name}"
    outdir.mkdir(parents=True, exist_ok=True)

    harm_paths = []
    for p in all_paths:
        raw = ad.read_h5ad(p)
        harm = run_h5adify_harmonization(raw, target_species="human", use_llm=use_llm)

        # final sanity
        if pd.Index(harm.var_names.astype(str)).has_duplicates:
            harm = collapse_duplicate_varnames_sum(harm)

        outp = outdir / ("harm_" + p.name)
        harm.write_h5ad(outp)
        harm_paths.append(outp)
        print(f"[OK] {method_name} Harmonized:", outp.name)
    return harm_paths, outdir

harm_paths_no_llm, OUT_NO_LLM = harmonize_all("no_llm", use_llm=False)

harm_paths_llm, OUT_LLM = (None, None)
if RUN_LLM:
    try:
        harm_paths_llm, OUT_LLM = harmonize_all("llm", use_llm=True)
    except Exception as e:
        print("[WARN] LLM run failed/skipped:", e)
        harm_paths_llm, OUT_LLM = None, None

# %% ==========================================================
# Cell 7 — Evaluate harmonization (metadata + sex + ortholog spot checks) for each method
# ==========================================================
EVAL_FIELDS = ["batch","sample","donor","domain","species","technology","sex"]

def evaluate_method(harm_paths: list[Path], method_name: str, method_outdir: Path):
    summary_rows = []
    for raw_p, harm_p in zip(all_paths, harm_paths):
        raw = ad.read_h5ad(raw_p)
        harm = ad.read_h5ad(harm_p)

        df_meta = evaluate_metadata_fields(raw, harm, EVAL_FIELDS)
        sex_acc, sex_cm = evaluate_sex(raw, harm)

        row = {
            "method": method_name,
            "file": raw_p.name,
            "sex_acc": sex_acc,
            **summarize_gene_namespace(harm),
        }
        for f in ["batch","donor","domain","species","technology","sex","sample"]:
            v = df_meta.loc[df_meta["field"] == f, "ari"]
            row[f"ari_{f}"] = float(v.values[0]) if len(v) else np.nan

        if "truth_mouse2human_spotcheck" in raw.uns:
            spot = gene_spotcheck_orthologs(harm, raw.uns["truth_mouse2human_spotcheck"])
            row["ortholog_spotcheck_recall"] = float(spot["present_expected"].mean())
        else:
            row["ortholog_spotcheck_recall"] = np.nan

        summary_rows.append(row)

        # save sex CM plot for every file (optional; can be heavy)
        if sex_cm is not None:
            fig = plt.figure(figsize=(5.2, 4.2))
            ax = fig.add_subplot(111)
            im = ax.imshow(sex_cm, interpolation="nearest")
            ax.set_title(f"Sex confusion\n{method_name}\n{raw_p.name}\nacc={sex_acc:.3f}")
            ax.set_xticks([0,1,2], ["female","male","unknown"], rotation=30, ha="right")
            ax.set_yticks([0,1,2], ["female","male","unknown"])
            for (i,j), v in np.ndenumerate(sex_cm):
                ax.text(j, i, str(v), ha="center", va="center", fontsize=9)
            fig.colorbar(im, ax=ax, fraction=0.05, pad=0.04)
            fig.tight_layout()
            fig.savefig(method_outdir / f"plot_sex_cm_{raw_p.stem}.png", dpi=170)
            plt.close(fig)

    summary = pd.DataFrame(summary_rows).sort_values("file")
    summary.to_csv(method_outdir / "summary_harmonization_metrics.csv", index=False)
    return summary

summary_no_llm = evaluate_method(harm_paths_no_llm, "no_llm", OUT_NO_LLM)
print(summary_no_llm.head())

summary_llm = None
if harm_paths_llm is not None:
    summary_llm = evaluate_method(harm_paths_llm, "llm", OUT_LLM)
    print(summary_llm.head())

# %% ==========================================================
# Cell 8 — Improved ARI plots (wide + readable)
# ==========================================================
def plot_ari_wide(summary_df: pd.DataFrame, cols: list[str], title: str, outpath: Path):
    df = summary_df.copy()
    df["short_file"] = df["file"].str.replace(".h5ad","", regex=False).str.replace("harm_","", regex=False)
    vals = df[["short_file"] + cols].set_index("short_file")

    n = vals.shape[0]
    fig_w = max(16, n * 0.75)
    fig_h = 6.5
    fig = plt.figure(figsize=(fig_w, fig_h))
    ax = fig.add_subplot(111)

    vals.plot(kind="bar", ax=ax)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel("ARI")
    ax.set_title(title)
    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    ax.tick_params(axis="x", labelrotation=45)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")

    # put legend outside
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9, frameon=False)
    fig.tight_layout()
    fig.savefig(outpath, dpi=180, bbox_inches="tight")
    plt.show()

plot_ari_wide(
    summary_no_llm,
    cols=["ari_batch","ari_donor","ari_domain","ari_species","ari_technology","ari_sex"],
    title="Metadata harmonization correctness (ARI) per file — no LLM",
    outpath=OUT_NO_LLM / "plot_metadata_ari.png",
)

if summary_llm is not None:
    plot_ari_wide(
        summary_llm,
        cols=["ari_batch","ari_donor","ari_domain","ari_species","ari_technology","ari_sex"],
        title="Metadata harmonization correctness (ARI) per file — LLM",
        outpath=OUT_LLM / "plot_metadata_ari.png",
    )

# %% ===========================================================
# Cell 9 — Merge, batch-correct, and compute batch-correction metrics (pre vs post) for each method
# ===========================================================
import scanpy as sc

def prepare_for_ml(adata: ad.AnnData, max_genes=2000):
    a = adata.copy()
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)

    if a.n_vars > max_genes:
        X = a.X
        if sp.issparse(X):
            mean = np.asarray(X.mean(axis=0)).ravel()
            mean2 = np.asarray(X.power(2).mean(axis=0)).ravel()
            v = mean2 - mean**2
        else:
            v = X.var(axis=0)
        top = np.argsort(-v)[:max_genes]
        a = a[:, top].copy()
    return a

def merge_group(paths: list[Path], kind: str):
    ads = [ad.read_h5ad(p) for p in paths]
    batch_key = "batch" if "batch" in ads[0].obs.columns else "h5adify_batch"
    merged = merge_datasets(ads, join="inner", batch_key=batch_key, harmonize_first=False)
    merged.uns["merge_kind"] = kind
    return merged, batch_key

def cross_group_classifier(adata: ad.AnnData, label_key: str, group_key: str, train_group, test_group, n_svd=50):
    obs = adata.obs
    if label_key not in obs.columns or group_key not in obs.columns:
        return None

    train_idx = np.where(obs[group_key].astype(str).values == str(train_group))[0]
    test_idx  = np.where(obs[group_key].astype(str).values == str(test_group))[0]
    if len(train_idx) < 200 or len(test_idx) < 200:
        return None

    Xtr = adata.X[train_idx]
    Xte = adata.X[test_idx]
    ytr = obs[label_key].astype(str).values[train_idx]
    yte = obs[label_key].astype(str).values[test_idx]

    svd = TruncatedSVD(n_components=min(n_svd, adata.n_vars-1), random_state=0)
    Ztr = svd.fit_transform(Xtr)
    Zte = svd.transform(Xte)

    clf = LogisticRegression(max_iter=600, n_jobs=-1)
    clf.fit(Ztr, ytr)
    pred = clf.predict(Zte)

    return {
        "train_group": str(train_group),
        "test_group": str(test_group),
        "acc": float(accuracy_score(yte, pred)),
        "macro_f1": float(f1_score(yte, pred, average="macro")),
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
    }

def run_merge_and_batch_metrics(harm_paths: list[Path], method_outdir: Path, method_name: str):
    harm_A = [p for p in harm_paths if p.name.startswith("harm_simA_")]
    harm_B = [p for p in harm_paths if p.name.startswith("harm_simB_")]
    harm_C = [p for p in harm_paths if p.name.startswith("harm_simC_")]

    merged_A, batch_key_A = merge_group(harm_A, "HumanBrain_scRNA")
    merged_B, batch_key_B = merge_group(harm_B, "GBM_PDX")
    merged_C, batch_key_C = merge_group(harm_C, "MouseSpatial_multitech")

    mA = prepare_for_ml(merged_A, max_genes=1500)
    mB = prepare_for_ml(merged_B, max_genes=1500)
    mC = prepare_for_ml(merged_C, max_genes=1500)

    # -----------------------------
    # Pick bio + sex labels
    # -----------------------------
    bioA = "truth_celltype" if "truth_celltype" in mA.obs.columns else None
    bioB = "truth_celltype" if "truth_celltype" in mB.obs.columns else None
    bioC = "truth_region"   if "truth_region"   in mC.obs.columns else None

    sexA = "truth_sex" if "truth_sex" in mA.obs.columns else None
    sexB = "truth_sex" if "truth_sex" in mB.obs.columns else None
    sexC = "truth_sex" if "truth_sex" in mC.obs.columns else None

    # -----------------------------
    # NEW: robust technology key for C
    # We create a canonical __tech_for_eval even if h5adify didn't write technology fields.
    # -----------------------------
    tech_candidates = [
        "technology", "h5adify_technology",
        "platform", "h5adify_platform",
        "assay", "h5adify_assay",
        "truth_technology",  # always present in your simulations
    ]
    tech_key_C = ensure_obs_alias(mC, "__tech_for_eval", tech_candidates, cast_str=True)

    # -----------------------------
    # Metrics PRE
    # -----------------------------
    met_rows = []
    met_rows.append({"method": method_name, "dataset": "A_pre",
                     **batch_correction_metrics(mA, batch_key_A, bioA, sexA, k=KNN_K, subsample=METRICS_SUBSAMPLE)})
    met_rows.append({"method": method_name, "dataset": "B_pre",
                     **batch_correction_metrics(mB, batch_key_B, bioB, sexB, k=KNN_K, subsample=METRICS_SUBSAMPLE)})

    # For C_pre, if we have a tech key use it; otherwise fall back to batch_key_C
    if tech_key_C is None:
        print(f"[WARN] ({method_name}) No technology-like key found for C; using batch key '{batch_key_C}' for C metrics.")
        met_rows.append({"method": method_name, "dataset": "C_pre",
                         **batch_correction_metrics(mC, batch_key_C, bioC, sexC, k=KNN_K, subsample=METRICS_SUBSAMPLE)})
        group_key_C_for_eval = batch_key_C
    else:
        met_rows.append({"method": method_name, "dataset": "C_pre",
                         **batch_correction_metrics(mC, tech_key_C, bioC, sexC, k=KNN_K, subsample=METRICS_SUBSAMPLE)})
        group_key_C_for_eval = tech_key_C

    # -----------------------------
    # ComBat (A by batch; C by technology if possible)
    # -----------------------------
    try:
        mA_cc = filter_zero_variance_genes(mA, eps=0.0)
        mA_combat = batch_correct_merged(mA_cc, batch_key=batch_key_A, method="combat")
    except Exception as e:
        print(f"[WARN] ComBat A failed ({method_name}):", e)
        mA_combat = None

    mC_combat = None
    if tech_key_C is None:
        print(f"[WARN] ComBat C skipped ({method_name}): no technology key available.")
    else:
        try:
            mC_cc = filter_zero_variance_genes(mC, eps=0.0)
            # IMPORTANT: correct by technology, not by section/batch
            mC_combat = batch_correct_merged(mC_cc, batch_key=tech_key_C, method="combat")
        except Exception as e:
            print(f"[WARN] ComBat C failed ({method_name}):", e)
            mC_combat = None

    # -----------------------------
    # Metrics POST
    # -----------------------------
    if mA_combat is not None:
        met_rows.append({"method": method_name, "dataset": "A_post_combat",
                         **batch_correction_metrics(mA_combat, batch_key_A, bioA, sexA, k=KNN_K, subsample=METRICS_SUBSAMPLE)})

    if mC_combat is not None:
        # ensure canonical tech key exists on corrected object too
        tech_key_C2 = ensure_obs_alias(mC_combat, "__tech_for_eval", tech_candidates, cast_str=True) or tech_key_C
        met_rows.append({"method": method_name, "dataset": "C_post_combat",
                         **batch_correction_metrics(mC_combat, tech_key_C2, bioC, sexC, k=KNN_K, subsample=METRICS_SUBSAMPLE)})

    met_df = pd.DataFrame(met_rows)
    met_df.to_csv(method_outdir / "summary_batch_correction_metrics.csv", index=False)

    # -----------------------------
    # Cross-group prediction checks
    # -----------------------------
    ml_rows = []

    # A pre/post (train on one batch, test on another)
    bA = list(pd.unique(mA.obs[batch_key_A].astype(str)))
    if len(bA) >= 2 and bioA is not None:
        r = cross_group_classifier(mA, label_key=bioA, group_key=batch_key_A,
                                   train_group=bA[0], test_group=bA[1])
        if r: ml_rows.append({"method": method_name, "dataset": "A_pre", **r})

    if mA_combat is not None and bioA is not None:
        bA2 = list(pd.unique(mA_combat.obs[batch_key_A].astype(str)))
        if len(bA2) >= 2:
            r = cross_group_classifier(mA_combat, label_key=bioA, group_key=batch_key_A,
                                       train_group=bA2[0], test_group=bA2[1])
            if r: ml_rows.append({"method": method_name, "dataset": "A_post_combat", **r})

    # C pre/post (train on one technology, test on another), using group_key_C_for_eval
    tC = list(pd.unique(mC.obs[group_key_C_for_eval].astype(str)))
    if len(tC) >= 2 and bioC is not None:
        r = cross_group_classifier(mC, label_key=bioC, group_key=group_key_C_for_eval,
                                   train_group=tC[0], test_group=tC[1])
        if r: ml_rows.append({"method": method_name, "dataset": "C_pre", **r})

    if mC_combat is not None and bioC is not None:
        # pick the same grouping key name if possible
        gkey_post = group_key_C_for_eval
        if gkey_post not in mC_combat.obs.columns:
            # if combat dropped it, rebuild canonical
            gkey_post = ensure_obs_alias(mC_combat, "__tech_for_eval", tech_candidates, cast_str=True)
        if gkey_post is not None:
            tC2 = list(pd.unique(mC_combat.obs[gkey_post].astype(str)))
            if len(tC2) >= 2:
                r = cross_group_classifier(mC_combat, label_key=bioC, group_key=gkey_post,
                                           train_group=tC2[0], test_group=tC2[1])
                if r: ml_rows.append({"method": method_name, "dataset": "C_post_combat", **r})

    ml_df = pd.DataFrame(ml_rows)
    ml_df.to_csv(method_outdir / "summary_prediction_metrics.csv", index=False)

    # return bundles for UMAP section
    return met_df, ml_df, (mA, mA_combat, mC, mC_combat, batch_key_A, group_key_C_for_eval)

met_no_llm, ml_no_llm, bundle_no_llm = run_merge_and_batch_metrics(harm_paths_no_llm, OUT_NO_LLM, "no_llm")
print(met_no_llm)

met_llm, ml_llm, bundle_llm = (None, None, None)
if harm_paths_llm is not None:
    met_llm, ml_llm, bundle_llm = run_merge_and_batch_metrics(harm_paths_llm, OUT_LLM, "llm")
    print(met_llm)

# %% ===========================================================
# Cell 10 — Plot batch correction metrics (pre vs post, and no_llm vs llm)
# ===========================================================
def plot_metrics(met_df: pd.DataFrame, title: str, outpath: Path):
    # choose a subset of key metrics to keep readable
    key_cols = [c for c in ["sil_batch","sil_bio","knn_batch_mixing","knn_bio_purity","sil_batch_female","sil_batch_male"] if c in met_df.columns]
    plot_df = met_df[["dataset"] + key_cols].set_index("dataset")

    fig = plt.figure(figsize=(14, 5.5))
    ax = fig.add_subplot(111)
    plot_df.plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    ax.tick_params(axis="x", labelrotation=25)
    for t in ax.get_xticklabels():
        t.set_ha("right")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9, frameon=False)
    fig.tight_layout()
    fig.savefig(outpath, dpi=180, bbox_inches="tight")
    plt.show()

plot_metrics(met_no_llm, "Batch correction metrics — no LLM", OUT_NO_LLM / "plot_batch_correction_metrics.png")

if met_llm is not None:
    plot_metrics(met_llm, "Batch correction metrics — LLM", OUT_LLM / "plot_batch_correction_metrics.png")

# %% ===========================================================
# Cell 11 — UMAP quicklooks (optional): pre and post ComBat for A and C
# ===========================================================
def safe_colors(adata: ad.AnnData, wanted: list[str]) -> list[str]:
    """Keep only color keys that exist in obs."""
    return [c for c in wanted if c in adata.obs.columns]

def quick_umap(adata: ad.AnnData, color: list[str], title: str, outpath: Path):
    import scanpy as sc
    a = adata.copy()
    sc.pp.highly_variable_genes(a, n_top_genes=min(2000, a.n_vars), flavor="seurat_v3")
    a = a[:, a.var["highly_variable"]].copy()
    sc.pp.scale(a, max_value=10)
    sc.tl.pca(a, n_comps=50)
    sc.pp.neighbors(a, n_neighbors=15, n_pcs=30)
    sc.tl.umap(a)
    sc.pl.umap(a, color=color, show=False, wspace=0.4, title=title)
    plt.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.close()

# --- no-LLM bundle ---
mA, mA_combat, mC, mC_combat, batch_key_A, group_key_C = bundle_no_llm

try:
    cols = safe_colors(mA, [batch_key_A, "truth_celltype", "truth_sex"])
    quick_umap(mA, color=cols, title="Sim A — no LLM (pre)", outpath=OUT_NO_LLM/"umap_simA_pre.png")
except Exception as e:
    print("[WARN] UMAP A pre (no_llm) failed:", e)

if mA_combat is not None:
    try:
        cols = safe_colors(mA_combat, [batch_key_A, "truth_celltype", "truth_sex"])
        quick_umap(mA_combat, color=cols, title="Sim A — no LLM (post ComBat)", outpath=OUT_NO_LLM/"umap_simA_post_combat.png")
    except Exception as e:
        print("[WARN] UMAP A post (no_llm) failed:", e)

try:
    cols = safe_colors(mC, [group_key_C, "truth_region", "truth_sex"])
    quick_umap(mC, color=cols, title="Sim C — no LLM (pre)", outpath=OUT_NO_LLM/"umap_simC_pre.png")
except Exception as e:
    print("[WARN] UMAP C pre (no_llm) failed:", e)

if mC_combat is not None:
    try:
        cols = safe_colors(mC_combat, [group_key_C, "truth_region", "truth_sex"])
        quick_umap(mC_combat, color=cols, title="Sim C — no LLM (post ComBat)", outpath=OUT_NO_LLM/"umap_simC_post_combat.png")
    except Exception as e:
        print("[WARN] UMAP C post (no_llm) failed:", e)

# --- LLM bundle ---
mA, mA_combat, mC, mC_combat, batch_key_A, group_key_C = bundle_llm

try:
    cols = safe_colors(mA, [batch_key_A, "truth_celltype", "truth_sex"])
    quick_umap(mA, color=cols, title="Sim A — LLM (pre)", outpath=OUT_LLM/"umap_simA_pre.png")
except Exception as e:
    print("[WARN] UMAP A pre (llm) failed:", e)

if mA_combat is not None:
    try:
        cols = safe_colors(mA_combat, [batch_key_A, "truth_celltype", "truth_sex"])
        quick_umap(mA_combat, color=cols, title="Sim A — LLM (post ComBat)", outpath=OUT_LLM/"umap_simA_post_combat.png")
    except Exception as e:
        print("[WARN] UMAP A post (llm) failed:", e)

try:
    cols = safe_colors(mC, [group_key_C, "truth_region", "truth_sex"])
    quick_umap(mC, color=cols, title="Sim C — LLM (pre)", outpath=OUT_LLM/"umap_simC_pre.png")
except Exception as e:
    print("[WARN] UMAP C pre (llm) failed:", e)

if mC_combat is not None:
    try:
        cols = safe_colors(mC_combat, [group_key_C, "truth_region", "truth_sex"])
        quick_umap(mC_combat, color=cols, title="Sim C — LLM (post ComBat)", outpath=OUT_LLM/"umap_simC_post_combat.png")
    except Exception as e:
        print("[WARN] UMAP C post (llm) failed:", e)

print("[OK] Wrote UMAPs to:", OUT_NO_LLM, "and", OUT_LLM)

# %% [markdown]
# ## Outputs written
# For each method folder:
# - summary_harmonization_metrics.csv
# - plot_metadata_ari.png
# - summary_batch_correction_metrics.csv
# - plot_batch_correction_metrics.png
# - summary_prediction_metrics.csv
# - sex confusion matrices per file
# - optional UMAPs (pre/post)

print("[OK] no_llm outputs:", OUT_NO_LLM)
if OUT_LLM is not None:
    print("[OK] llm outputs:", OUT_LLM)
